# CIFAKE Image Classification - Random Forest Model

This notebook implements a Random Forest classifier to distinguish between REAL and FAKE (AI-generated) images using the CIFAKE dataset.

## Environment & Reproducibility
* **Dataset:** CIFAKE: Real and AI-Generated Synthetic Images.
* **Data Split:** 80,000 (Train) / 20,000 (Validation) / 20,000 (Test). Stratified splitting is used to maintain class balance.
* **Random State:** `random_state = 42` is set globally to ensure all splits and model initializations are reproducible.
* **Key Libraries:** Python 3.10+, `scikit-learn`, `numpy`, `pandas`, `matplotlib`, `scikit-image`, `opencv-python`, `PyWavelets`.

## How to Run This Notebook
1. **Setup & Data Download:** Run the initial cells to mount Google Drive and download the CIFAKE dataset via Kaggle API.
2. **Feature Extraction:** The notebook extracts multiple handcrafted features (DCT, HOG, LBP, GLCM, Wavelet, Color Histogram). **Note:** These cells are computationally expensive and can take up to 1-2 hours to complete depending on Colab's allocated resources.
3. **Model Training (E1 - E10):** The notebook systematically trains 10 Random Forest configurations with different feature combinations.
4. **Evaluation & Selection:** Validation accuracy is compared to select the best configuration (E9). The final chosen model is then evaluated on the completely independent Test set, followed by error analysis and ROC-AUC evaluation.
5. **Save State:** The final model and feature matrices are saved to Google Drive for future use without needing to retrain or re-extract features.

##Import the needed libraries


In [ ]:
# Import packages for data handling
import pandas as pd
import numpy as np

# Import packages for plotting
import matplotlib.pyplot as plt

# Import model selection tools
from sklearn.model_selection import train_test_split

# Import evaluation metrics
from sklearn import metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Import classifiers
from sklearn.ensemble import RandomForestClassifier

# Display all dataframe columns
pd.set_option('display.max_columns', None)

# Make results reproducible
random_state = 42

##Download the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)

##Link the drive with Colab


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
cd /content/drive/MyDrive/AT2-IntroToAI

##Create train, validate, test datasets

In [ ]:
from pathlib import Path
# Use the path variable returned by kagglehub
dataset = Path(path) / 'train'
realImages = sorted(list((dataset/'REAL').glob('*.jpg')))
fakeImages = sorted(list((dataset/'FAKE').glob("*.jpg")))

image_paths = realImages + fakeImages
labels = ['REAL'] * len(realImages) + ['FAKE'] * len(fakeImages)
# Split into training data and temporary data
X_train, X_val, y_train, y_val = train_test_split(image_paths, labels, test_size=0.2, stratify = labels, random_state=random_state)

#Test set
testpath = Path(path) / 'test'
realTestImages = sorted(list((testpath/'REAL').glob('*.jpg')))
fakeTestImages = sorted(list((testpath/'FAKE').glob("*.jpg")))
X_test = realTestImages + fakeTestImages
y_test = ['REAL'] * len(realTestImages) + ['FAKE'] * len(fakeTestImages)

##Extract features from the images using Discrete Cosine Transform


In [ ]:
import cv2
from scipy.fftpack import dct
from tqdm import tqdm

def scipy_dct2d(img):
    return dct(dct(img.T, norm='ortho').T, norm='ortho')

def extract_dct_features(image_paths, size=(32, 32), num_coefficients=100):
    features = []
    for path in tqdm(image_paths, desc="Extracting DCT features"):
        # Read Gray Scale Images
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to read image: {path}")

        # Calculate DCT 2D
        dct_coefs = scipy_dct2d(img.astype(np.float32))

        # Extract low-frequency
        k = int(np.sqrt(num_coefficients))
        low_freq = dct_coefs[:k, :k].flatten()
        features.append(low_freq)

    return np.array(features)

In [ ]:
# Extracting features from the datasets
print("Extracting features for Training set...")
X_train_dct = extract_dct_features(X_train, size=(32, 32), num_coefficients=100)

print("Extracting features for Validation set...")
X_val_dct = extract_dct_features(X_val, size=(32, 32), num_coefficients=100)

print("Extracting features for Testing set...")
X_test_dct = extract_dct_features(X_test, size=(32, 32), num_coefficients=100)

print(X_train_dct.shape)
print(X_val_dct.shape)
print(X_test_dct.shape)

In [ ]:
print(np.isfinite(X_train_dct).all())
print(np.isfinite(X_val_dct).all())
print(np.isfinite(X_test_dct).all())
print(X_train_dct.shape[0] == len(y_train))
print(X_val_dct.shape[0] == len(y_val))
print(X_test_dct.shape[0] == len(y_test))

## Helper function for validation

In [ ]:
def evaluate_model(developed_model, model_name, X_eval_data, y_eval_true):
    # Predict labels using the provided evaluation features
    y_preds = developed_model.predict(X_eval_data)

    # Calculate standard classification metrics
    accuracy_test = accuracy_score(y_eval_true, y_preds)
    precision_test = metrics.precision_score(y_eval_true, y_preds, average='weighted', zero_division=0)
    recall_test = metrics.recall_score(y_eval_true, y_preds, average='weighted', zero_division=0)
    f1_test = metrics.f1_score(y_eval_true, y_preds, average='weighted', zero_division=0)

    print('=' * 70)
    print('Selected model:', model_name)
    print('Final validation performance')
    print('=' * 70)
    print('Validation accuracy:', round(accuracy_test, 4))
    print('Validation precision, weighted:', round(precision_test, 4))
    print('Validation recall, weighted:', round(recall_test, 4))
    print('Validation F1-score, weighted:', round(f1_test, 4))

    print()
    print('Classification report:')
    print(classification_report(y_eval_true, y_preds, target_names=class_names, zero_division=0))

    # Display confusion matrix
    cm = confusion_matrix(y_eval_true, y_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(5, 4), dpi=100)
    disp.plot(ax=ax)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

    return y_preds

### Experiment E1: Random Forest + GrayScale DCT Features

In [ ]:
# Define class labels
class_names = ['FAKE', 'REAL']

# Initiate and train the model
rf_model = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_model.fit(X_train_dct, y_train)

# Evaluate the model on the validation set using DCT features
developed_model = rf_model
y_preds = evaluate_model(developed_model, 'Random Forest (Original DCT Features)', X_val_dct, y_val)

###Extracting Features using Histogram of Oriented Gradients


In [ ]:
from skimage.feature import hog
def extract_hog_features(image_paths, size=(32, 32)):
    features = []
    for path in tqdm(image_paths, desc="Extracting HOG features"):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to read image: {path}")
        img_resized = cv2.resize(img, size)

        # Extract HOG features
        hog_feats = hog(
            img_resized,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            visualize=False
        )
        features.append(hog_feats)

    return np.array(features)

In [ ]:
# Perform HOG feature extraction for Training and Validation sets
print("Extracting HOG for Training set...")
X_train_hog = extract_hog_features(X_train)

print("Extracting HOG for Validation set...")
X_val_hog = extract_hog_features(X_val)

### Experiment E2: Random Forest + features extracted from HOG

In [ ]:
# Initialize and train the Random Forest model on HOG features
rf_hog = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_hog.fit(X_train_hog, y_train)

# Evaluate results by passing X_val_hog and y_val
print("Evaluating Random Forest + HOG:")
y_preds_hog = evaluate_model(rf_hog, "Random Forest (HOG Features)", X_val_hog, y_val)

### Experiment E3: Random Forest model using combined features from DCT + HOG

In [ ]:
# Concatenate DCT and HOG features together
X_train_combined = np.hstack((X_train_dct, X_train_hog))
X_val_combined = np.hstack((X_val_dct, X_val_hog))

# Initialize and train the model
rf_combined = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_combined.fit(X_train_combined, y_train)

# Evaluate the model using the updated evaluate_model function
print("Evaluating Random Forest + DCT + HOG:")
y_preds_comb = evaluate_model(rf_combined, "Random Forest (DCT + HOG Features)", X_val_combined, y_val)

### Extract Features using Local Binary Pattern (LBP)


In [ ]:
from skimage.feature import local_binary_pattern
import cv2
import numpy as np
from tqdm import tqdm

def extract_lbp_features(image_paths, size=(32, 32), P=8, R=1, method='uniform', n_bins=10):
    features = []
    for path in tqdm(image_paths, desc="Extracting LBP features"):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to read image: {path}")
        img_resized = cv2.resize(img, size)

        # Compute LBP representation
        lbp = local_binary_pattern(img_resized, P, R, method=method)

        # Calculate histogram as the feature vector representation
        # For 'uniform' method, the maximum number of labels is P + 2
        hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))

        # Normalize the histogram so that the sum is 1
        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-7)

        features.append(hist)

    return np.array(features)

In [ ]:
# Extract LBP features for Training and Validation sets
print("Extracting LBP for Training set...")
X_train_lbp = extract_lbp_features(X_train)

print("Extracting LBP for Validation set...")
X_val_lbp = extract_lbp_features(X_val)

### Experiment E4: Random Forest + LBP

In [ ]:
# Initialize and train the Random Forest model on LBP features
rf_lbp = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_lbp.fit(X_train_lbp, y_train)

# Evaluate E4 model
print("Evaluating E4: Random Forest + LBP:")
y_preds_lbp = evaluate_model(rf_lbp, "Random Forest (LBP Features)", X_val_lbp, y_val)

### Experiment E5: Random Forest + DCT + HOG + LBP


In [ ]:
# Concatenate all three features: DCT, HOG, and LBP
X_train_all = np.hstack((X_train_dct, X_train_hog, X_train_lbp))
X_val_all = np.hstack((X_val_dct, X_val_hog, X_val_lbp))

# Initialize and train the model
rf_all = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_all.fit(X_train_all, y_train)

# Evaluate E5 model
print("Evaluating E5: Random Forest + DCT + HOG + LBP:")
y_preds_all = evaluate_model(rf_all, "Random Forest (DCT + HOG + LBP Features)", X_val_all, y_val)

### Install needed Libraries for Wavelet


In [ ]:
!pip install PyWavelets

### Define feature-extracting functions for 4 more techniques  (GLCM, Wavelet, Color Histogram, RGB DCT)

In [ ]:
from skimage.feature import graycomatrix, graycoprops
import pywt

def extract_glcm_features(image_paths, size=(32, 32)):
    features = []
    for path in tqdm(image_paths, desc="Extracting GLCM features"):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to read image: {path}")
        img_resized = cv2.resize(img, size)

        # Compute GLCM matrix
        glcm = graycomatrix(img_resized, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)

        # Extract texture statistical features
        contrast = graycoprops(glcm, 'contrast').flatten()
        correlation = graycoprops(glcm, 'correlation').flatten()
        energy = graycoprops(glcm, 'energy').flatten()
        homogeneity = graycoprops(glcm, 'homogeneity').flatten()

        feats = np.concatenate([contrast, correlation, energy, homogeneity])
        features.append(feats)
    return np.array(features)

def extract_wavelet_features(image_paths, size=(32, 32), wavelet='db1', level=1):
    features = []
    for path in tqdm(image_paths, desc="Extracting Wavelet features"):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to read image: {path}")
        img_resized = cv2.resize(img, size)


        # 2D Discrete Wavelet Transform (DWT 2D)
        coeffs = pywt.dwt2(img_resized, wavelet)
        cA, (cH, cV, cD) = coeffs

        # Extract statistical features from sub-bands
        feats = np.array([
            np.mean(cA), np.std(cA),
            np.mean(cH), np.std(cH),
            np.mean(cV), np.std(cV),
            np.mean(cD), np.std(cD)
        ])
        features.append(feats)
    return np.array(features)

def extract_color_hist_features(image_paths, size=(32, 32), bins=(8, 8, 8)):
    features = []
    for path in tqdm(image_paths, desc="Extracting Color Histograms"):
        img = cv2.imread(str(path))
        if img is None:
            raise ValueError(f"Failed to read image: {path}")
        img_resized = cv2.resize(img, size)

        # Compute RGB color histogram
        hist = cv2.calcHist([img_resized], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
        cv2.normalize(hist, hist)
        features.append(hist.flatten())
    return np.array(features)

def extract_rgb_dct_features(image_paths, size=(32, 32), num_coefficients=100):
    features = []
    k = int(np.sqrt(num_coefficients))
    for path in tqdm(image_paths, desc="Extracting RGB DCT features"):
        img = cv2.imread(str(path))
        if img is None:
            raise ValueError(f"Failed to read image: {path}")
        img_resized = cv2.resize(img, size)

        # Compute DCT independently on three color channels
        dct_channels = []
        for i in range(3):
            ch = img_resized[:, :, i].astype(np.float32)
            coefs = scipy_dct2d(ch)
            low_freq = coefs[:k, :k].flatten()
            dct_channels.append(low_freq)

        features.append(np.concatenate(dct_channels))
    return np.array(features)

### Extracting Features using GLCM, WAVELET,...

In [ ]:
# Extracting new features
print("Extracting GLCM...")
X_train_glcm = extract_glcm_features(X_train)
X_val_glcm = extract_glcm_features(X_val)

print("Extracting Wavelet...")
X_train_wavelet = extract_wavelet_features(X_train)
X_val_wavelet = extract_wavelet_features(X_val)

print("Extracting Color Histograms...")
X_train_color = extract_color_hist_features(X_train)
X_val_color = extract_color_hist_features(X_val)

print("Extracting RGB DCT...")
X_train_rgb_dct = extract_rgb_dct_features(X_train)
X_val_rgb_dct = extract_rgb_dct_features(X_val)

### Experiment E6: Random Forest + E5 (DCT+HOG+LBP) + GLCM

In [ ]:
X_train_e6 = np.hstack((X_train_all, X_train_glcm))
X_val_e6 = np.hstack((X_val_all, X_val_glcm))

rf_e6 = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_e6.fit(X_train_e6, y_train)

print("Evaluating E6 (DCT + HOG + LBP + GLCM):")
y_preds_e6 = evaluate_model(rf_e6, "Random Forest E6 (DCT + HOG + LBP + GLCM)", X_val_e6, y_val)

### Experiment E7: Random Forest + E5 (DCT+HOG+LBP) + Wavelet

In [ ]:
X_train_e7 = np.hstack((X_train_all, X_train_wavelet))
X_val_e7 = np.hstack((X_val_all, X_val_wavelet))

rf_e7 = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_e7.fit(X_train_e7, y_train)

print("Evaluating E7 (DCT + HOG + LBP + Wavelet):")
y_preds_e7 = evaluate_model(rf_e7, "Random Forest E7 (DCT + HOG + LBP + Wavelet)", X_val_e7, y_val)

### Experiment E8: Random Forest + E5 + GLCM + Wavelet

In [ ]:
X_train_e8 = np.hstack((X_train_all, X_train_glcm, X_train_wavelet))
X_val_e8 = np.hstack((X_val_all, X_val_glcm, X_val_wavelet))

rf_e8 = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_e8.fit(X_train_e8, y_train)

print("Evaluating E8 (DCT + HOG + LBP + GLCM + Wavelet):")
y_preds_e8 = evaluate_model(rf_e8, "Random Forest E8 (DCT + HOG + LBP + GLCM + Wavelet)", X_val_e8, y_val)

### Experiment E9: Random Forest + E8 + Color Histogram

In [ ]:
X_train_e9 = np.hstack((X_train_e8, X_train_color))
X_val_e9 = np.hstack((X_val_e8, X_val_color))

rf_e9 = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_e9.fit(X_train_e9, y_train)

print("Evaluating E9 (E8 + Color Histogram):")
y_preds_e9 = evaluate_model(rf_e9, "Random Forest E9 (E8 + Color Histogram)", X_val_e9, y_val)

### Experiment E10: Substitute Grayscale DCT by RGB DCT combining features extracted from HOG + LBP + GLCM + Wavelet

In [ ]:
X_train_e10 = np.hstack((X_train_rgb_dct, X_train_hog, X_train_lbp, X_train_glcm, X_train_wavelet))
X_val_e10 = np.hstack((X_val_rgb_dct, X_val_hog, X_val_lbp, X_val_glcm, X_val_wavelet))

rf_e10 = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
rf_e10.fit(X_train_e10, y_train)

print("Evaluating E10 (RGB DCT + HOG + LBP + GLCM + Wavelet):")
y_preds_e10 = evaluate_model(rf_e10, "Random Forest E10 (RGB DCT + HOG + LBP + GLCM + Wavelet)", X_val_e10, y_val)

## Validation Performance Summary (E1 - E10)
To provide a clear empirical basis for our final model selection, the following table and chart summarize the Validation Accuracy across all 10 experiments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

# Calculate validation accuracies based on saved predictions
val_accuracies = {
    'E1 (DCT)': accuracy_score(y_val, y_preds),
    'E2 (HOG)': accuracy_score(y_val, y_preds_hog),
    'E3 (DCT + HOG)': accuracy_score(y_val, y_preds_comb),
    'E4 (LBP)': accuracy_score(y_val, y_preds_lbp),
    'E5 (DCT + HOG + LBP)': accuracy_score(y_val, y_preds_all),
    'E6 (E5 + GLCM)': accuracy_score(y_val, y_preds_e6),
    'E7 (E5 + Wavelet)': accuracy_score(y_val, y_preds_e7),
    'E8 (E5 + GLCM + Wavelet)': accuracy_score(y_val, y_preds_e8),
    'E9 (E8 + Color Hist)': accuracy_score(y_val, y_preds_e9),
    'E10 (RGB DCT + others)': accuracy_score(y_val, y_preds_e10)
}

# Create a DataFrame for nice formatting
df_summary = pd.DataFrame(list(val_accuracies.items()), columns=['Experiment Configuration', 'Validation Accuracy'])
df_summary['Validation Accuracy (%)'] = ((df_summary['Validation Accuracy'] * 100) + 1e-9).round(2)

# Display the Summary Table
display(df_summary[['Experiment Configuration', 'Validation Accuracy (%)']])

# Plotting the results
plt.figure(figsize=(12, 6))

# Highlight E9 with a distinct color, others neutral
colors = ['#d62728' if 'E9' in x else '#1f77b4' for x in df_summary['Experiment Configuration']]

ax = sns.barplot(x='Validation Accuracy (%)', y='Experiment Configuration', data=df_summary, palette=colors, hue='Experiment Configuration', legend=False)

# Customizing the plot
plt.title('Validation Accuracy Comparison across E1-E10', fontsize=14, fontweight='bold')
plt.xlabel('Validation Accuracy (%)', fontsize=12)
plt.ylabel('Experiment Configuration', fontsize=12)
plt.xlim(0, 100) # Adjust x-axis limits to 0-100 to show true proportions
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Add exact percentage labels on the bars
for p in ax.patches:
    ax.annotate(f'{p.get_width():.2f}%',
                (p.get_width() + 1.0, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=10, color='black', fontweight='bold')

plt.tight_layout()
plt.show()

### Key Insights from Validation Performance
Based on the chart and results table, we can draw 3 main insights:
1. **Feature Fusion:** Combining features yields significant improvements. Specifically, the E3 configuration (DCT + HOG) achieves an accuracy of 80.14%, substantially higher than using DCT alone in E1 (75.48%).
2. **Texture Features:** Adding texture features like GLCM and Wavelet to the E5 configuration improved accuracy from 80.71% to 86.46% in E8.
3. **Best Configuration:** The E9 model achieves the highest result (88.99%) when color distribution information (Color Histogram) is added, whereas E10, despite substituting with more complex components, could not surpass this configuration.


## Helper Function for Test dataset

In [ ]:
def evaluate_model1(developed_model, model_name, X_eval_data, y_eval_true):
    # Predict labels using the provided evaluation features
    y_preds = developed_model.predict(X_eval_data)

    # Calculate standard classification metrics
    accuracy_test = accuracy_score(y_eval_true, y_preds)
    precision_test = metrics.precision_score(y_eval_true, y_preds, average='weighted', zero_division=0)
    recall_test = metrics.recall_score(y_eval_true, y_preds, average='weighted', zero_division=0)
    f1_test = metrics.f1_score(y_eval_true, y_preds, average='weighted', zero_division=0)

    print('=' * 70)
    print('Selected model:', model_name)
    print('Final test performance')
    print('=' * 70)
    print('Test accuracy:', round(accuracy_test, 4))
    print('Test precision, weighted:', round(precision_test, 4))
    print('Test recall, weighted:', round(recall_test, 4))
    print('Test F1-score, weighted:', round(f1_test, 4))

    print()
    print('Classification report:')
    print(classification_report(y_eval_true, y_preds, target_names=class_names, zero_division=0))

    # Display confusion matrix
    cm = confusion_matrix(y_eval_true, y_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(5, 4), dpi=100)
    disp.plot(ax=ax)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

    return y_preds

## Extracting Features from different techniques

In [ ]:
# Extract necessary test features for E9 and E10
print("Extracting GLCM for Test set...")
X_test_glcm = extract_glcm_features(X_test)

print("Extracting Wavelet for Test set...")
X_test_wavelet = extract_wavelet_features(X_test)

print("Extracting Color Histograms for Test set...")
X_test_color = extract_color_hist_features(X_test)

print("Extracting RGB DCT for Test set...")
X_test_rgb_dct = extract_rgb_dct_features(X_test)

In [ ]:
# Extract missing HOG and LBP features for the Test set
print("Extracting HOG for Test set...")
X_test_hog = extract_hog_features(X_test)

print("Extracting LBP for Test set...")
X_test_lbp = extract_lbp_features(X_test)

## Comparing E9 and E10 models on the test set

In [ ]:
# Construct combined test feature vectors
# E9 uses: DCT + HOG + LBP + GLCM + Wavelet + Color Histogram
X_test_all = np.hstack((X_test_dct, X_test_hog, X_test_lbp))
X_test_e8 = np.hstack((X_test_all, X_test_glcm, X_test_wavelet))
X_test_e9 = np.hstack((X_test_e8, X_test_color))

# E10 uses: RGB DCT + HOG + LBP + GLCM + Wavelet
X_test_e10 = np.hstack((X_test_rgb_dct, X_test_hog, X_test_lbp, X_test_glcm, X_test_wavelet))

# Evaluate E9 on the Test set using the existing evaluate_model function
print("Evaluating E9 (E8 + Color Histogram) on Test Set:")
y_test_preds_e9 = evaluate_model1(rf_e9, "Random Forest E9 (E8 + Color Histogram)", X_test_e9, y_test)

# Evaluate E10 on the Test set
print("Evaluating E10 (RGB DCT + HOG + LBP + GLCM + Wavelet) on Test Set:")
y_test_preds_e10 = evaluate_model1(rf_e10, "Random Forest E10 (RGB DCT + HOG + LBP + GLCM + Wavelet)", X_test_e10, y_test)

## Final Model Selection: Why E9?

**1. Selection Criterion & Evidence:**
Our model selection process relies on **Validation Accuracy** and **Weighted F1-score**. The validation set was strictly used to guide model selection, while the test set was kept entirely independent for the final evaluation to prevent data leakage. Among all 10 experiments, **Experiment E9** achieved the highest Validation Accuracy of **88.99%**, which is higher than E8 (86.46%) and E10 (87.44%). Because E9 yielded the best performance on the validation set, it was chosen as the final model.

**2. Technical Rationale:**
Empirical results demonstrate that feature fusion positively contributes to the model's performance. E9 leverages a combination of six complementary feature sets:
*   **DCT and Wavelet** for frequency domain information.
*   **HOG** for edge structures and gradients.
*   **LBP and GLCM** for texture properties.
*   **Color Histogram** for color distribution information.
Specifically, adding the Color Histogram to the E8 configuration improved the validation accuracy from 86.46% to 88.99%.

**3. Final Verification on Test Set:**
After selecting E9 based on the validation results, we used the strictly independent Test Set for final evaluation. E9 achieved an **89.41% Test Accuracy** and a **0.894 Weighted F1-score**. The gap between Validation Accuracy (88.99%) and Test Accuracy (89.41%) is small (0.42 percentage points), demonstrating relatively consistent performance across the CIFAKE evaluation sets.

However, while it performs well on unseen CIFAKE data, this does not guarantee generalization to images generated by other AI tools or external real-world images. Despite this limitation, E9 remains the best-performing model among the evaluated candidates from our experiments to implement in the final User Interface.

In [ ]:
import joblib

# Path to save the model
model_path = 'rf_e9.joblib'

# Save the rf_e9 model
joblib.dump(rf_e9, model_path)
print(f"Model E9 saved successfully at: {model_path}\n")

# Verify n_features_in_ and classes_
print("=== MODEL INFORMATION VERIFICATION ===")
print(f"Number of input features (n_features_in_): {rf_e9.n_features_in_}")
print(f"List of classification classes (classes_): {rf_e9.classes_}")

## Error Analysis on Test Set (E9 Model)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Identify misclassified indices
misclassified_indices = [i for i, (true, pred) in enumerate(zip(y_test, y_test_preds_e9)) if true != pred]

# Separate into two error types
fake_as_real = [i for i in misclassified_indices if y_test[i] == 'FAKE' and y_test_preds_e9[i] == 'REAL']
real_as_fake = [i for i in misclassified_indices if y_test[i] == 'REAL' and y_test_preds_e9[i] == 'FAKE']

print(f"Total misclassifications on Test Set: {len(misclassified_indices)}")
print(f"FAKE misclassified as REAL: {len(fake_as_real)}")
print(f"REAL misclassified as FAKE: {len(real_as_fake)}")

# Helper function to plot misclassified images
def plot_misclassified(indices, title, num_images=5):
    plt.figure(figsize=(15, 3))
    for i, idx in enumerate(indices[:num_images]):
        img_path = X_test[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(1, num_images, i + 1)
        plt.imshow(img)
        plt.title(f"True: {y_test[idx]}\nPred: {y_test_preds_e9[idx]}")
        plt.axis('off')
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.1)
    plt.show()

# Visualize some examples
if len(fake_as_real) > 0:
    plot_misclassified(fake_as_real, "Examples of FAKE images misclassified as REAL")

if len(real_as_fake) > 0:
    plot_misclassified(real_as_fake, "Examples of REAL images misclassified as FAKE")


### Qualitative Error Analysis

Based on the error distribution (985 FAKE images misclassified as REAL and 1,134 REAL images misclassified as FAKE) and the visual examples, we can formulate several hypotheses regarding the limitations of the E9 model:

**1. FAKE Misclassified as REAL (False Negatives for FAKE - 985 images, 46.48% of errors):**
* **Practical Implications:** In an AI-detection system, a False Negative means synthetic content slips through undetected. This failure could allow misleading or fake information to spread as authentic content.
* **Hypothesis on Feature Distribution:** The fake images in the CIFAKE dataset (generated by Stable Diffusion) often achieve high visual realism. We hypothesize that for many of these misclassified cases, the generated images possess characteristics that are highly realistic, causing their extracted hand-crafted features (like GLCM, LBP, or Wavelet) to closely overlap with the feature distribution of real images. While the Random Forest model successfully learned a decision boundary, it struggles to perfectly separate these overlapping cases without more complex semantic information.

**2. REAL Misclassified as FAKE (False Positives for FAKE - 1,134 images, 53.52% of errors):**
* **Practical Implications:** A False Positive means authentic images are falsely flagged as AI-generated. This could damage trust in genuine content or wrongly penalize creators. Although this error is slightly more frequent (an 11.34% error rate for the REAL class vs. 9.85% for the FAKE class), stating that the model is "overly sensitive" to specific texture or gradient anomalies remains a hypothesis that requires further empirical verification.
* **Hypothesis on Natural Anomalies:** We hypothesize that natural artifacts—such as motion blur, low-light sensor noise, or heavy color grading—could be possible contributing factors. If present, these factors might significantly alter the **Color Histogram** or create abnormal gradient vectors in **HOG**. When these feature vectors deviate too much from the expected feature distribution learned from the training data, the model may mistakenly classify them as FAKE. However, a deeper analysis of the specific feature values within these 1,134 misclassified images is needed to confirm this.

**Future Recommendations:**
Currently, this model relies entirely on **hand-crafted features** (color distribution, texture, frequency). A promising next step for research is to evaluate **Deep Learning architectures (such as CNNs or Vision Transformers)** to see if automatic, hierarchical feature extraction can improve classification capabilities compared to manual feature engineering. (Note: As another member of our team is independently evaluating a CNN model on this dataset, these findings should ideally be cross-referenced with their actual CNN results to confirm whether deep learning overcomes these specific feature overlap issues). However, it must be noted that given the low 32x32 resolution of the CIFAKE dataset, many semantic details are inherently obscured, and CNNs do not automatically understand physical or geometric anomalies. Therefore, deep learning should be approached as an exploratory evaluation rather than a guaranteed solution to E9's limitations.

### ROC-AUC and ROC Curve for Random Forest E9

To comprehensively evaluate the E9 model's ability to distinguish between FAKE and REAL images across various classification thresholds (rather than just at the default 0.5 threshold), we analyze the **Receiver Operating Characteristic (ROC) Curve** and the **Area Under the Curve (ROC-AUC)**.

In [ ]:
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib.pyplot as plt

# 1. Encode labels: FAKE -> 1 (Positive Class), REAL -> 0 (Negative Class)
y_test_binary = np.array([1 if label == 'FAKE' else 0 for label in y_test])

# 2. Get the predicted probabilities for the positive class (FAKE -> index 0)
y_pred_proba_e9_fake = rf_e9.predict_proba(X_test_e9)[:, 0]

# 3. Calculate False Positive Rate (FPR), True Positive Rate (TPR), and Thresholds
fpr, tpr, thresholds = roc_curve(y_test_binary, y_pred_proba_e9_fake)

# 4. Calculate the ROC-AUC score
roc_auc = auc(fpr, tpr)

# 5. Plot the ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Model E9 ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve - Model E9\n(Positive Class: FAKE)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ROC-AUC Score for Model E9 on Test Set (Positive Class: FAKE): {roc_auc:.4f}")

### ROC-AUC Evaluation & Limitations

**1. Excellent Discriminative Ability (ROC-AUC = 0.9603):**
The Random Forest E9 model achieves a high ROC-AUC score of 0.9603 on the CIFAKE Test Set. This indicates that the model has approximately a 96.03% probability of ranking a randomly chosen FAKE image higher than a randomly chosen REAL image. It demonstrates a robust ability to separate the two classes effectively across various decision thresholds.

**2. ROC Curve Characteristics:**
The plotted ROC curve is positioned near the top-left corner, far exceeding the diagonal random guess baseline (AUC = 0.5). This visually confirms that the model has a strong discriminative ability. However, it is worth noting that as we adjust the threshold to prioritize a higher True Positive Rate (TPR), the False Positive Rate (FPR) will inevitably increase.

**3. Limitations:**
While a 0.9603 AUC score is highly indicative of strong performance on the CIFAKE dataset, it does not guarantee similar generalization on images generated by other external AI models (e.g., Midjourney, DALL-E) or out-of-distribution real-world data. Furthermore, it is critical to note that if any threshold optimization is needed in the future to strictly balance False Positives and False Negatives for a real-world application, that tuning must be performed on the Validation Set rather than this Test Set.

## Save Final Feature Matrices (E9)
To avoid spending hours re-extracting features in the future, we will save the finalized feature matrices and labels for the best-performing E9 configuration directly to Google Drive.

In [ ]:
import numpy as np
import os

# Define the save path in Google Drive
save_path = '/content/drive/MyDrive/AT2-IntroToAI/'

# Ensure the directory exists
os.makedirs(save_path, exist_ok=True)

print("Saving E9 feature matrices and labels to Google Drive...")

# Save Training Set
np.save(os.path.join(save_path, 'X_train_e9.npy'), X_train_e9)
np.save(os.path.join(save_path, 'y_train.npy'), np.array(y_train))

# Save Validation Set
np.save(os.path.join(save_path, 'X_val_e9.npy'), X_val_e9)
np.save(os.path.join(save_path, 'y_val.npy'), np.array(y_val))

# Save Test Set
np.save(os.path.join(save_path, 'X_test_e9.npy'), X_test_e9)
np.save(os.path.join(save_path, 'y_test.npy'), np.array(y_test))

print(f"Saved successfully to {save_path}!")